In [0]:
WITH all_trips AS (
    SELECT * FROM yellow_tripdata_january
    UNION ALL
    SELECT * FROM yellow_tripdata_february
    UNION ALL
    SELECT * FROM yellow_tripdata_march
),

weather_flags AS (
    SELECT
        DATE                                                    AS weather_date,
        PRCP,
        CASE
            WHEN PRCP > 25 OR COALESCE(WT16, 0) = 1 THEN 1
            ELSE 0
        END                                                     AS is_rainy_day
    FROM `ny-weather-data`
),

clean_trips AS (
    SELECT
        t.tpep_pickup_datetime,
        t.PULocationID,
        t.DOLocationID,
        t.passenger_count,
        t.fare_amount,
        t.total_amount,
        DATE(t.tpep_pickup_datetime)                            AS pickup_date,
        w.is_rainy_day,
        -- Snap each pickup time to the start of its 15-minute slot.
        -- 900 = 15 minutes × 60 seconds.
        -- UNIX_TIMESTAMP() converts to seconds since 1970.
        -- FLOOR(x/900)*900 rounds down to the nearest 15-min boundary.
        -- TIMESTAMP_SECONDS() converts the result back to a readable timestamp.
        -- If TIMESTAMP_SECONDS is unavailable, replace with:
        -- CAST(FROM_UNIXTIME(FLOOR(UNIX_TIMESTAMP(t.tpep_pickup_datetime)/900)*900) AS TIMESTAMP)
        TIMESTAMP_SECONDS(
            FLOOR(UNIX_TIMESTAMP(t.tpep_pickup_datetime) / 240) * 240
        )                                                       AS time_bucket_4min
    FROM all_trips t
    INNER JOIN weather_flags w
        ON DATE(t.tpep_pickup_datetime) = w.weather_date
    WHERE
        t.fare_amount       > 0
        AND t.total_amount  > 0
        AND t.trip_distance > 0
        AND t.passenger_count BETWEEN 1 AND 6
),

-- Group by the combination that defines a shareable opportunity:
-- same pickup zone + same dropoff zone + same 15-minute window.
-- HAVING COUNT(*) >= 2 keeps only groups where sharing is actually possible.
shareable_groups AS (
    SELECT
        pickup_date,
        is_rainy_day,
        PULocationID,
        DOLocationID,
        time_bucket_4min,
        COUNT(*)                                                AS trips_in_group,
        SUM(passenger_count)                                    AS total_people_in_group,
        ROUND(AVG(total_amount), 2)                             AS avg_fare_per_trip
    FROM clean_trips
    GROUP BY
        pickup_date,
        is_rainy_day,
        PULocationID,
        DOLocationID,
        time_bucket_4min
    HAVING COUNT(*) >= 2
)

SELECT
    CASE WHEN is_rainy_day = 1 THEN 'Rainy Day' ELSE 'Dry Day'
    END                                                         AS weather_type,
    COUNT(*)                                                    AS shareable_groups_found,
    SUM(trips_in_group)                                         AS total_trips_that_could_share,
    ROUND(AVG(trips_in_group), 2)                               AS avg_group_size,
    MAX(trips_in_group)                                         AS largest_group_found,
    -- Breakdown by group size: most will be pairs, some trios, rare 4+
    SUM(CASE WHEN trips_in_group = 2  THEN 1 ELSE 0 END)        AS groups_of_exactly_2,
    SUM(CASE WHEN trips_in_group = 3  THEN 1 ELSE 0 END)        AS groups_of_exactly_3,
    SUM(CASE WHEN trips_in_group >= 4 THEN 1 ELSE 0 END)        AS groups_of_4_or_more,
    ROUND(AVG(avg_fare_per_trip), 2)                            AS avg_fare_in_shareable_trips_usd
FROM shareable_groups
GROUP BY is_rainy_day
ORDER BY is_rainy_day DESC;